# SRC-001 Scenario Telemetry Audit

## Purpose

Audit the scenario-level TXT telemetry files contained in SRC-001 to determine
their structure, consistency, quality, temporal information, and usefulness as
scenario evidence.

## Out of Scope

This notebook does not:

- modify or clean raw telemetry files;
- analyze PCAP captures;
- perform cross-source alignment;
- train machine-learning models;
- evaluate RCA performance.

## 1. Setup

Locate the scenario-level TXT telemetry files associated with SRC-001 and
prepare them for structural inspection without modifying the raw source.

In [23]:
import pandas as pd
import json
import numpy as np

from src.data import inventory

In [2]:
SOURCE_ID = "SRC-001"

inventory = inventory.DataInventory(source_id=SOURCE_ID)

RAW_DATA_PATH = inventory.raw_data_path

In [3]:
txt_files = sorted(
    path
    for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == ".txt"
)

txt_files

[PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/benign.txt'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/ddos_icmp_hping3.txt'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/ftp_hydra.txt'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/icmp_nping.txt'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/portscan_masscan.txt'),
 PosixPath('/Users/emiliogarcialopez/projectGitHub/distributed-agentic-ai-platform/data/raw/SRC-001_netslab_5g_oran_idd/web_xss.txt')]

In [4]:
if not txt_files:
    raise FileNotFoundError(
        f"No TXT files found in: {RAW_DATA_PATH}"
    )

In [5]:
scenario_files = pd.DataFrame(
    {
        "scenario": [path.stem for path in txt_files],
        "file_name": [path.name for path in txt_files],
        "size_mb": [
            path.stat().st_size / (1024 ** 2)
            for path in txt_files
        ],
    }
)

scenario_files

,scenario,file_name,size_mb
0,benign,benign.txt,0.805487
1,ddos_icmp_hping3,ddos_icmp_hping3.txt,0.511050
2,ftp_hydra,ftp_hydra.txt,0.207704
3,icmp_nping,icmp_nping.txt,0.356653
4,portscan_masscan,portscan_masscan.txt,0.388638
5,web_xss,web_xss.txt,0.084647


### Finding

The SRC-001 raw source contains multiple scenario-level TXT telemetry files.

Each file is associated with a scenario through its filename and can be
inspected independently before evaluating schema and content consistency across
scenarios.

No telemetry content has been modified at this stage.

## 2. File Structure

Inspect the physical readability and parsing structure of the scenario-level
TXT telemetry files before performing content-level analysis.

In [6]:
readability_results = []

for path in txt_files:
    try:
        with path.open("r", encoding="utf-8", errors="strict") as file:
            first_line = file.readline()

        readability_results.append(
            {
                "file_name": path.name,
                "readable": True,
                "first_line": first_line.strip(),
            }
        )

    except Exception as error:
        readability_results.append(
            {
                "file_name": path.name,
                "readable": False,
                "first_line": None,
                "error": str(error),
            }
        )

readability_results = pd.DataFrame(readability_results)

readability_results

,file_name,readable,first_line
0,benign.txt,True,"{""ue_id"": 2, ""timestamp"": 1751374264245, ""cell..."
1,ddos_icmp_hping3.txt,True,"{""ue_id"": 1, ""timestamp"": 1751364899238, ""cell..."
2,ftp_hydra.txt,True,"{""ue_id"": 1, ""timestamp"": 1751030178797, ""cell..."
3,icmp_nping.txt,True,"{""ue_id"": 1, ""timestamp"": 1751036039759, ""cell..."
4,portscan_masscan.txt,True,"{""ue_id"": 1, ""timestamp"": 1750872369844, ""cell..."
5,web_xss.txt,True,"{""ue_id"": 1, ""timestamp"": 1751282193281, ""cell..."


In [7]:
for path in txt_files:
    print(f"\n--- {path.name} ---")

    with path.open("r", encoding="utf-8", errors="replace") as file:
        for _ in range(3):
            print(file.readline().rstrip())


--- benign.txt ---
{"ue_id": 2, "timestamp": 1751374264245, "cellid": 11111111, "in_sync": 1, "rnti": "097a", "dlBytes": 541944552, "dlMcs": 23, "dlBler": 0.064554, "ulBytes": 821528571, "ulMcs": 11, "ulBler": 0.161862, "ri": 1, "pmi_0": 0, "pmi_1": 0, "phr": 25, "pcmax": 23, "rsrq": -10.5, "sinr": 23.0, "rsrp": -94, "rssi": -75.8, "cqi": 15, "pucchSnr": 20.0, "puschSnr": 22.0}
{"ue_id": 15, "timestamp": 1751374264245, "cellid": 11111111, "in_sync": 1, "rnti": "de81", "dlBytes": 14016622, "dlMcs": 9, "dlBler": 0.0, "ulBytes": 69276604, "ulMcs": 9, "ulBler": 0.070186, "ri": 1, "pmi_0": 0, "pmi_1": 0, "phr": 23, "pcmax": 25, "rsrq": -11.0, "sinr": 11.0, "rsrp": -101, "rssi": -73.4, "cqi": 15, "pucchSnr": 18.0, "puschSnr": 19.0}
{"ue_id": 2, "timestamp": 1751374265245, "cellid": 11111111, "in_sync": 1, "rnti": "097a", "dlBytes": 541974006, "dlMcs": 24, "dlBler": 0.178451, "ulBytes": 821535047, "ulMcs": 11, "ulBler": 0.069676, "ri": 1, "pmi_0": 0, "pmi_1": 0, "phr": 25, "pcmax": 23, "rsrq

In [8]:
scenario_inventory = pd.DataFrame(
    {
        "scenario": [path.stem for path in txt_files],
        "file_name": [path.name for path in txt_files],
        "size_mb": [
            path.stat().st_size / (1024 ** 2)
            for path in txt_files
        ],
        "readable": [
            path.name in readability_results.loc[
                readability_results["readable"],
                "file_name",
            ].values
            for path in txt_files
        ],
    }
)

scenario_inventory

,scenario,file_name,size_mb,readable
0,benign,benign.txt,0.805487,True
1,ddos_icmp_hping3,ddos_icmp_hping3.txt,0.511050,True
2,ftp_hydra,ftp_hydra.txt,0.207704,True
3,icmp_nping,icmp_nping.txt,0.356653,True
4,portscan_masscan,portscan_masscan.txt,0.388638,True
5,web_xss,web_xss.txt,0.084647,True


### Finding

Six scenario-level TXT files were identified and all were successfully read.

The files correspond to the following scenarios:

- `benign`
- `ddos_icmp_hping3`
- `ftp_hydra`
- `icmp_nping`
- `portscan_masscan`
- `web_xss`

Initial inspection shows that the files are not delimiter-based text tables.
Instead, each line contains a JSON-formatted telemetry record.

The observed records include fields such as UE identifiers, timestamps,
cell identifiers, synchronization state, RNTI, uplink/downlink counters,
radio-quality measurements, modulation information, and SNR-related metrics.

The files are therefore suitable for structured parsing as line-delimited JSON
records. Schema consistency across scenarios must be verified in the next
section.

## 3. Telemetry Structure

Verify that scenario-level JSON telemetry records can be parsed consistently
and determine whether all scenario files share the same schema.

In [9]:
scenario_data = {}

for path in txt_files:
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line in file:
            records.append(json.loads(line))

    scenario_data[path.stem] = pd.DataFrame(records)

In [10]:
for scenario, dataframe in scenario_data.items():
    print(scenario, dataframe.shape)

benign (2357, 23)
ddos_icmp_hping3 (1500, 23)
ftp_hydra (601, 23)
icmp_nping (1037, 23)
portscan_masscan (1127, 23)
web_xss (245, 23)


In [11]:
scenario_schemas = {
    scenario: set(dataframe.columns)
    for scenario, dataframe in scenario_data.items()
}

scenario_schemas

{'benign': {'cellid',
  'cqi',
  'dlBler',
  'dlBytes',
  'dlMcs',
  'in_sync',
  'pcmax',
  'phr',
  'pmi_0',
  'pmi_1',
  'pucchSnr',
  'puschSnr',
  'ri',
  'rnti',
  'rsrp',
  'rsrq',
  'rssi',
  'sinr',
  'timestamp',
  'ue_id',
  'ulBler',
  'ulBytes',
  'ulMcs'},
 'ddos_icmp_hping3': {'cellid',
  'cqi',
  'dlBler',
  'dlBytes',
  'dlMcs',
  'in_sync',
  'pcmax',
  'phr',
  'pmi_0',
  'pmi_1',
  'pucchSnr',
  'puschSnr',
  'ri',
  'rnti',
  'rsrp',
  'rsrq',
  'rssi',
  'sinr',
  'timestamp',
  'ue_id',
  'ulBler',
  'ulBytes',
  'ulMcs'},
 'ftp_hydra': {'cellid',
  'cqi',
  'dlBler',
  'dlBytes',
  'dlMcs',
  'in_sync',
  'pcmax',
  'phr',
  'pmi_0',
  'pmi_1',
  'pucchSnr',
  'puschSnr',
  'ri',
  'rnti',
  'rsrp',
  'rsrq',
  'rssi',
  'sinr',
  'timestamp',
  'ue_id',
  'ulBler',
  'ulBytes',
  'ulMcs'},
 'icmp_nping': {'cellid',
  'cqi',
  'dlBler',
  'dlBytes',
  'dlMcs',
  'in_sync',
  'pcmax',
  'phr',
  'pmi_0',
  'pmi_1',
  'pucchSnr',
  'puschSnr',
  'ri',
  'rnti',
  

In [12]:
reference_schema = next(iter(scenario_schemas.values()))

schema_consistent = all(
    schema == reference_schema
    for schema in scenario_schemas.values()
)

schema_consistent

True

In [13]:
for scenario, schema in scenario_schemas.items():
    missing = reference_schema - schema
    extra = schema - reference_schema

    if missing or extra:
        print(f"\n{scenario}")
        print("Missing:", sorted(missing))
        print("Extra:", sorted(extra))

### Finding

All six scenario telemetry files were successfully parsed as line-delimited
JSON records.

Each scenario uses the same 23-field schema, with no missing or additional
fields detected between files.

Record counts vary by scenario, but the telemetry structure is consistent
across the complete TXT subset.

The files are therefore structurally compatible for joint analysis.

In [14]:
dtype_summary = {}

for scenario, dataframe in scenario_data.items():
    dtype_summary[scenario] = {
        column: str(dtype)
        for column, dtype in dataframe.dtypes.items()
    }

dtype_summary

{'benign': {'ue_id': 'int64',
  'timestamp': 'int64',
  'cellid': 'int64',
  'in_sync': 'int64',
  'rnti': 'str',
  'dlBytes': 'int64',
  'dlMcs': 'int64',
  'dlBler': 'float64',
  'ulBytes': 'int64',
  'ulMcs': 'int64',
  'ulBler': 'float64',
  'ri': 'int64',
  'pmi_0': 'int64',
  'pmi_1': 'int64',
  'phr': 'int64',
  'pcmax': 'int64',
  'rsrq': 'float64',
  'sinr': 'float64',
  'rsrp': 'int64',
  'rssi': 'float64',
  'cqi': 'int64',
  'pucchSnr': 'float64',
  'puschSnr': 'float64'},
 'ddos_icmp_hping3': {'ue_id': 'int64',
  'timestamp': 'int64',
  'cellid': 'int64',
  'in_sync': 'int64',
  'rnti': 'str',
  'dlBytes': 'int64',
  'dlMcs': 'int64',
  'dlBler': 'float64',
  'ulBytes': 'int64',
  'ulMcs': 'int64',
  'ulBler': 'float64',
  'ri': 'int64',
  'pmi_0': 'int64',
  'pmi_1': 'int64',
  'phr': 'int64',
  'pcmax': 'int64',
  'rsrq': 'float64',
  'sinr': 'float64',
  'rsrp': 'int64',
  'rssi': 'float64',
  'cqi': 'int64',
  'pucchSnr': 'float64',
  'puschSnr': 'float64'},
 'ftp_hydr

In [15]:
reference_dtypes = next(iter(dtype_summary.values()))

dtype_consistent = all(
    dtypes == reference_dtypes
    for dtypes in dtype_summary.values()
)

dtype_consistent

True

In [16]:
for scenario, dtypes in dtype_summary.items():
    differences = {
        column: (reference_dtypes[column], dtypes[column])
        for column in reference_dtypes
        if reference_dtypes[column] != dtypes[column]
    }

    if differences:
        print(f"\n{scenario}")
        print(differences)

### Finding

All six scenario telemetry files share the same 23-field schema and the same
Pandas data types.

No schema or dtype inconsistencies were detected between scenarios.

The TXT telemetry subset is therefore structurally homogeneous and suitable for
combined analysis across scenarios.

## 4. Data Quality

Evaluate whether the scenario telemetry contains missing values, duplicate
records, non-informative fields, or structurally invalid values that could
limit its use as evidence in the DAICP MVP.

In [17]:
telemetry_all = pd.concat(
    [
        dataframe.assign(scenario=scenario)
        for scenario, dataframe in scenario_data.items()
    ],
    ignore_index=True,
)

raw_fields = sorted(reference_schema)

telemetry_all.shape

(6867, 24)

In [18]:
missing_summary = pd.DataFrame(
    {
        "field": raw_fields,
        "missing_count": [
            telemetry_all[column].isna().sum()
            for column in raw_fields
        ],
        "missing_pct": [
            telemetry_all[column].isna().mean() * 100
            for column in raw_fields
        ],
    }
)

missing_summary[
    missing_summary["missing_count"] > 0
].sort_values("missing_count", ascending=False)

,field,missing_count,missing_pct


In [19]:
duplicate_mask = telemetry_all.duplicated(
    subset=raw_fields,
    keep="first",
)

duplicate_count = duplicate_mask.sum()
duplicate_pct = duplicate_count / len(telemetry_all) * 100

print("Total records:", len(telemetry_all))
print("Duplicate records:", duplicate_count)
print(f"Duplicate percentage: {duplicate_pct:.4f}%")

Total records: 6867
Duplicate records: 0
Duplicate percentage: 0.0000%


In [20]:
duplicate_by_scenario = (
    telemetry_all.loc[duplicate_mask]
    ["scenario"]
    .value_counts()
    .rename("duplicate_count")
)

duplicate_by_scenario

Series([], Name: duplicate_count, dtype: int64)

In [21]:
constant_fields = [
    column
    for column in raw_fields
    if telemetry_all[column].nunique(dropna=False) <= 1
]

constant_fields

['cellid', 'in_sync', 'pmi_0', 'pmi_1', 'ri']

In [24]:
numeric_fields = telemetry_all[
    raw_fields
].select_dtypes(include="number").columns.tolist()

infinite_counts = {
    column: int(np.isinf(telemetry_all[column]).sum())
    for column in numeric_fields
}

{
    column: count
    for column, count in infinite_counts.items()
    if count > 0
}

{}

In [25]:
byte_fields = [
    column
    for column in raw_fields
    if "bytes" in column.lower()
]

negative_byte_counts = {
    column: int((telemetry_all[column] < 0).sum())
    for column in byte_fields
}

negative_byte_counts

{'dlBytes': 0, 'ulBytes': 0}

In [26]:
temporal_fields = [
    column
    for column in raw_fields
    if any(
        token in column.lower()
        for token in ("timestamp", "time", "date")
    )
]

timestamp_issues = {}

for column in temporal_fields:
    timestamp_issues[column] = {
        "missing": int(telemetry_all[column].isna().sum()),
        "non_positive": int((telemetry_all[column] <= 0).sum()),
        "infinite": int(np.isinf(telemetry_all[column]).sum()),
    }

timestamp_issues

{'timestamp': {'missing': 0, 'non_positive': 0, 'infinite': 0}}

### Finding

The combined TXT telemetry subset contains 6,867 records and 24 columns, where
23 columns come from the raw telemetry schema and one additional `scenario`
column is used only to preserve file provenance during the audit.

No missing values or exact duplicate records were detected.

Five constant fields were identified across the complete TXT subset:
`cellid`, `in_sync`, `pmi_0`, `pmi_1`, and `ri`.

No infinite values, negative byte counters, or structurally invalid timestamp
values were detected.

Overall, the scenario TXT telemetry shows good structural quality and is
suitable for continued temporal and scenario-level analysis.

## 5. Scenario Structure

Inspect how telemetry records are distributed across the available SRC-001
scenario files and verify that each scenario contributes usable observations.

In [27]:
scenario_coverage = pd.DataFrame(
    {
        "scenario": list(scenario_data.keys()),
        "record_count": [
            len(dataframe)
            for dataframe in scenario_data.values()
        ],
    }
)

scenario_coverage["percentage"] = (
    scenario_coverage["record_count"]
    / scenario_coverage["record_count"].sum()
    * 100
)

scenario_coverage = (
    scenario_coverage
    .sort_values("record_count", ascending=False)
    .reset_index(drop=True)
)

scenario_coverage

,scenario,record_count,percentage
0,benign,2357,34.323577
1,ddos_icmp_hping3,1500,21.843600
2,portscan_masscan,1127,16.411825
3,icmp_nping,1037,15.101209
4,ftp_hydra,601,8.752002
5,web_xss,245,3.567788


In [28]:
empty_scenarios = scenario_coverage[
    scenario_coverage["record_count"] == 0
]

empty_scenarios

,scenario,record_count,percentage


### Finding

All six scenario TXT files contain valid telemetry observations and no empty
scenario files were identified.

A total of 6,867 records are distributed across the scenarios:

- `benign`: 2,357 records (34.32%)
- `ddos_icmp_hping3`: 1,500 records (21.84%)
- `portscan_masscan`: 1,127 records (16.41%)
- `icmp_nping`: 1,037 records (15.10%)
- `ftp_hydra`: 601 records (8.75%)
- `web_xss`: 245 records (3.57%)

The scenario subset is therefore usable, but it is not balanced. `benign`
contains the largest share of observations, while `web_xss` is substantially
underrepresented.

This imbalance must be considered in later evaluation and cross-source
comparison.

## 6. Temporal Structure

Evaluate timestamp validity and temporal coverage across the scenario telemetry
files to determine whether they can support later cross-source alignment.

In [29]:
timestamp_summary = telemetry_all[temporal_fields].agg(
    ["count", "nunique", "min", "max"]
)

timestamp_summary

,timestamp
count,6867
nunique,4526
min,1750872369844
max,1751375357835


In [30]:
TIMESTAMP_FIELD = temporal_fields[0]

telemetry_all["timestamp_datetime"] = pd.to_datetime(
    telemetry_all[TIMESTAMP_FIELD],
    unit="ms",
    utc=True,
)

In [31]:
temporal_coverage = (
    telemetry_all
    .groupby("scenario")["timestamp_datetime"]
    .agg(["min", "max", "count", "nunique"])
    .reset_index()
)

temporal_coverage["duration"] = (
    temporal_coverage["max"]
    - temporal_coverage["min"]
)

temporal_coverage

,scenario,min,max,count,nunique,duration
0,benign,2025-07-01 12:51:04.245000+00:00,2025-07-01 13:09:17.835000+00:00,2357,966,0 days 00:18:13.590000
1,ddos_icmp_hping3,2025-07-01 10:14:59.238000+00:00,2025-07-01 10:23:18.308000+00:00,1500,550,0 days 00:08:19.070000
2,ftp_hydra,2025-06-27 13:16:18.797000+00:00,2025-06-27 13:26:18.797000+00:00,601,601,0 days 00:10:00
3,icmp_nping,2025-06-27 14:53:59.759000+00:00,2025-06-27 15:11:15.759000+00:00,1037,1037,0 days 00:17:16
4,portscan_masscan,2025-06-25 17:26:09.844000+00:00,2025-06-25 17:44:55.844000+00:00,1127,1127,0 days 00:18:46
5,web_xss,2025-06-30 11:16:33.281000+00:00,2025-06-30 11:20:37.281000+00:00,245,245,0 days 00:04:04


In [32]:
print("Start:", telemetry_all["timestamp_datetime"].min())
print("End:", telemetry_all["timestamp_datetime"].max())
print(
    "Coverage:",
    telemetry_all["timestamp_datetime"].max()
    - telemetry_all["timestamp_datetime"].min()
)

Start: 2025-06-25 17:26:09.844000+00:00
End: 2025-07-01 13:09:17.835000+00:00
Coverage: 5 days 19:43:07.991000


### Finding

The TXT telemetry contains 6,867 timestamped records with 4,526 distinct
timestamps.

No missing, non-positive, or infinite timestamp values were detected. The
timestamps are consistent with Unix epoch time expressed in milliseconds.

The analyzed scenario subset spans from 2025-06-25 17:26:09 UTC to
2025-07-01 13:09:17 UTC, providing approximately 5 days and 19 hours of
overall temporal coverage.

Each scenario preserves its own temporal interval, ranging from a few minutes
to approximately 18 minutes of telemetry.

Repeated timestamps are present, but no exact duplicate records were detected;
therefore, repeated timestamps alone are not treated as a data-quality issue.

The TXT telemetry preserves sufficient temporal information for later
scenario-level and cross-source alignment analysis.

## 7. Audit Findings

The audit of the SRC-001 scenario TXT telemetry produced the following findings:

- Six scenario-level TXT files were identified and all were successfully parsed
  as line-delimited JSON records.
- The six files share the same 23-field schema and consistent data types.
- A total of 6,867 telemetry records were available across the scenario subset.
- No missing values were detected.
- No exact duplicate records were detected.
- Five constant fields were identified across the complete TXT subset:
  `cellid`, `in_sync`, `pmi_0`, `pmi_1`, and `ri`.
- No infinite values, negative byte counters, or structurally invalid timestamp
  values were detected.
- All six scenarios contain usable telemetry observations.
- Scenario representation is imbalanced, with `benign` being the most
  represented scenario and `web_xss` the least represented.
- The telemetry preserves explicit timestamps for all records.
- The analyzed TXT subset spans approximately 5 days and 19 hours overall,
  with each scenario covering its own shorter telemetry interval.
- The preserved timestamps make the TXT files suitable for later temporal
  comparison with corresponding PCAP captures and other SRC-001 sources.

No raw telemetry records were modified during this audit.

## 8. TXT Telemetry Suitability Decision

The SRC-001 scenario TXT telemetry is suitable for use in the DAICP MVP as
scenario-level lower-layer evidence.

### Intended Use

The TXT telemetry will be used for:

- scenario-specific radio and lower-layer telemetry analysis;
- temporal inspection of individual attack and benign scenarios;
- comparison with corresponding PCAP captures;
- cross-source alignment with other SRC-001 artifacts;
- supporting RCA hypotheses with scenario-level telemetry evidence.

### Limitations

The TXT subset has the following limitations:

- scenario representation is imbalanced;
- several fields are constant across the analyzed subset;
- the files cover only a limited set of SRC-001 scenarios;
- the telemetry alone does not provide complete network or packet-level context.

### Decision

The TXT telemetry files are accepted for the DAICP MVP.

They provide structurally consistent, timestamped, scenario-specific telemetry
with no missing values, exact duplicates, or major structural quality issues.

Their main value is the ability to associate lower-layer telemetry with
specific scenarios and later compare it with the corresponding PCAP captures.

The TXT files should therefore be retained as a complementary evidence source,
not as a standalone source for Root Cause Analysis.